# Lab 01–03: Lakehouse Partitioning, Delta Log & Time Travel

**Z5008 Big Data Lab (MTech Core) · IIT Madras Zanzibar**

Partition pruning, MERGE upserts, and reproducible version snapshots.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/inyalala/iitmz-teaching-site/blob/main/courses/z5008/labs/BD01_Lakehouse_Parquet_Partitioning.ipynb)

Run cells top to bottom (`Runtime > Run all`). Spark steps include pure-Python fallbacks when executed outside a Spark/JVM container.

## 1. Medallion Partitioning, Delta Log Commits & Time Travel

A Lakehouse table maintains an immutable JSON transaction log (`_delta_log/0000...json`). Readers inspect the snapshot metadata to prune irrelevant partitions and query historical versions (`VERSION AS OF`).

In [1]:
from dataclasses import dataclass, asdict
import json
from pathlib import Path
import tempfile

@dataclass(frozen=True)
class TxRecord:
    tx_id: str
    region: str
    amount_usd: float

class MiniDeltaTable:
    def __init__(self, root):
        self.root = Path(root)
        self.log_dir = self.root / "_delta_log"
        self.log_dir.mkdir(parents=True, exist_ok=True)
        self.version = -1

    def commit(self, records):
        state = {r["tx_id"]: r for r in self.read_with_stats()[0]} if self.version >= 0 else {}
        for r in records:
            state[r["tx_id"]] = r
        self.version += 1
        grouped, files = {}, []
        for row in state.values():
            grouped.setdefault(row["region"], []).append(row)
        for reg, rows in sorted(grouped.items()):
            p = self.root / f"region={reg}"
            p.mkdir(parents=True, exist_ok=True)
            fn = f"part-v{self.version:04d}.json"
            (p / fn).write_text(json.dumps(rows), encoding="utf-8")
            files.append(f"region={reg}/{fn}")
        (self.log_dir / f"{self.version:020d}.json").write_text(json.dumps({"version": self.version, "files": files}), encoding="utf-8")
        return self.version

    def read_with_stats(self, version=None, region=None):
        v = self.version if version is None else version
        meta = json.loads((self.log_dir / f"{v:020d}.json").read_text(encoding="utf-8"))
        rows, scanned = [], 0
        for rel in meta["files"]:
            if region and not rel.startswith(f"region={region}/"):
                continue
            scanned += 1
            rows.extend(json.loads((self.root / rel).read_text(encoding="utf-8")))
        return rows, scanned

with tempfile.TemporaryDirectory() as tmp:
    t = MiniDeltaTable(tmp)
    v0 = t.commit([asdict(TxRecord("T1", "Zanzibar", 120.0)), asdict(TxRecord("T2", "Dar", 310.0))])
    v1 = t.commit([asdict(TxRecord("T2", "Dar", 355.0)), asdict(TxRecord("T3", "Nairobi", 90.0))])
    r_old, _ = t.read_with_stats(version=0)
    r_dar, files_read = t.read_with_stats(version=1, region="Dar")
    print(f"v0 total rows={len(r_old)} | v1 Dar rows={r_dar} (partition files scanned={files_read})")
    assert len(r_old) == 2 and r_dar[0]["amount_usd"] == 355.0 and files_read == 1


v0 total rows=2 | v1 Dar rows=[{'tx_id': 'T2', 'region': 'Dar', 'amount_usd': 355.0}] (partition files scanned=1)
